# Build Pocket-3D Features (chromophore-anchored 3D-spatial)

Driver for `build_pocket3d_features.py` — extracts ~80 dim chromophore-HETATM-anchored features from minimized PDBs.

**Why a new structural block?** Diagnostic + clustered benchmark proved the prior structural blocks (ESM-IF1 2048-dim, hand-crafted CA 55-dim) hurt MAE in every configuration. Both were anchored at backbone-CA and blind to the HETATM chromophore ligand itself. This block anchors at phenol-OH / imidazolinone centroid / phenol-ring centroid → 3D-spatial shells over residues that sequence-window pooling misses.

**Runtime.** CPU-only, ~3 min for 913 PDBs. Authored as `.ipynb` for parity with `build_struct_features.ipynb` and Drive-mount logic.

Output: `pocket3d_features.npz` written to Drive.

In [ ]:
# ── Cell 1: Environment detection, set paths, register repo on sys.path ───────
import os, sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR    = Path('/content/drive/Othercomputers/My Mac/FluorCode/data')
    STRUCT_DIR  = DATA_DIR / 'minimized'
    SEQ_DIR     = DATA_DIR
    OUT_DIR     = Path('/content/drive/MyDrive/FluorCode/struct_features')
    REPO        = Path('/content/drive/MyDrive/FluorCode_colab')
    MODULE_DIR  = REPO / 'model' / 'LoRA_ESM2_Structure'
else:
    # Local execution: edit these paths to match your setup
    REPO        = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("../..")
    DATA_DIR    = REPO / 'data' / 'structure'
    STRUCT_DIR  = DATA_DIR / 'minimized'
    SEQ_DIR     = REPO / 'data' / 'sequence'
    OUT_DIR     = Path(__file__).resolve().parent if "__file__" in dir() else Path(".")
    MODULE_DIR  = Path(__file__).resolve().parent if "__file__" in dir() else Path(".")
    print("Running locally. Edit DATA_DIR / STRUCT_DIR / OUT_DIR if paths don't match your setup.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(MODULE_DIR))

assert (SEQ_DIR / 'fp_embeddings_meta.csv').exists(), f'meta csv missing under {SEQ_DIR}'
n_pdb = len(list(STRUCT_DIR.glob('*_minimized.pdb'))) if STRUCT_DIR.exists() else 0
print('DATA_DIR   :', DATA_DIR)
print('STRUCT_DIR :', STRUCT_DIR, '—', n_pdb, 'PDBs')
print('OUT_DIR    :', OUT_DIR)

In [ ]:
# ── Cell 2: Imports + module reload ──────────────────────────────────────────────────────────────────
import importlib
import pocket3d_atom_tables
import build_pocket3d_features as bp3
importlib.reload(pocket3d_atom_tables)
importlib.reload(bp3)

# Override repo-relative paths to point at Drive
bp3.DATA_DIR   = DATA_DIR
bp3.SEQ_DIR    = SEQ_DIR
bp3.STRUCT_DIR = STRUCT_DIR
bp3.OUT_DIR    = OUT_DIR

print('Total feature dims:', len(bp3.ALL_FEATURE_NAMES))
print('  Block A (chrom chemistry) :', len(bp3.BLOCK_A_NAMES))
print('  Block B (OH 3D env)       :', len(bp3.BLOCK_B_NAMES))
print('  Block C (imid 3D env)     :', len(bp3.BLOCK_C_NAMES))
print('  Block D (barrel arch)     :', len(bp3.BLOCK_D_NAMES))

In [ ]:
# ── Cell 3: Smoke-test on one known FP before full run ────────────────────────────────────────────────────────
import numpy as np
for slug in ['egfp', 'mcherry', 'dsred', '10b']:
    pdb = STRUCT_DIR / f'{slug}_minimized.pdb'
    if not pdb.exists():
        print(f'  {slug}: PDB missing, skip'); continue
    struct = bp3.parse_pdb(pdb)
    if struct is None:
        print(f'  {slug}: parse failed'); continue
    nres = len(struct.ca_coords)
    n_chrom_atoms = len(struct.chrom_atoms)
    chrom_name = struct.chrom_resname
    has_oh = 'OH' in struct.chrom_atoms
    print(f'  {slug:10s}  n_res={nres:4d}  chrom={chrom_name}  n_chrom_atoms={n_chrom_atoms:2d}  has_OH={has_oh}')
    if has_oh:
        vec, hp, ac = bp3.extract_for_slug(slug, pdb_dir=STRUCT_DIR)
        print(f'              vec.shape={vec.shape}  has_pocket={hp}  atom_completeness={ac:.3f}  ||vec||={float(np.linalg.norm(vec)):.2f}')

In [ ]:
# ── Cell 4: Run full extraction across all 986 slugs ────────────────────────────────────────────────────────────
import time
t0 = time.time()
out = bp3.extract_all(SEQ_DIR / 'fp_embeddings_meta.csv', pdb_dir=STRUCT_DIR, verbose=True)
print(f'\nelapsed: {time.time() - t0:.1f}s')

In [ ]:
# ── Cell 5: Save to Drive ────────────────────────────────────────────────────────────────────────────────────────
import numpy as np
out_path = OUT_DIR / 'pocket3d_features.npz'
np.savez(out_path, **out)
print(f'Saved {out["features"].shape} → {out_path}')
print(f'  has_pocket  : {int(out["has_pocket"].sum())}/{len(out["slugs"])}')
print(f'  atom_completeness mean (over has_pocket=1): '
      f'{float(out["atom_completeness"][out["has_pocket"] == 1].mean()):.3f}')

In [ ]:
# ── Cell 6: Sanity — NaN check + per-feature non-degeneracy ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
d = np.load(out_path, allow_pickle=True)
X = d['features']
names = list(d['feature_names'])
hp = d['has_pocket'].astype(bool)

print('any NaN:', bool(np.isnan(X).any()))
print('any inf:', bool(np.isinf(X).any()))

# Per-feature stats restricted to has_pocket=1 rows
X_ok = X[hp]
df = pd.DataFrame({
    'feature': names,
    'mean':    X_ok.mean(axis=0),
    'std':     X_ok.std(axis=0),
    'min':     X_ok.min(axis=0),
    'max':     X_ok.max(axis=0),
    'pct_zero':(X_ok == 0).mean(axis=0),
})
degenerate = df[(df['std'] < 1e-6) | (df['pct_zero'] > 0.9)]
print(f'\nDegenerate features (std<1e-6 OR >90% zeros): {len(degenerate)}/{len(df)}')
if len(degenerate):
    print(degenerate.to_string(index=False))
print('\nFirst 20 features (mean / std / pct_zero):')
print(df.head(20).to_string(index=False))

In [ ]:
# ── Cell 7: Distribution histograms per block ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

blocks = {'A': [], 'B': [], 'C': [], 'D': []}
for j, n in enumerate(names):
    blocks[n[0]].append(j)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
for ax, (block, idx) in zip(axes, blocks.items()):
    if not idx: continue
    vmat = X_ok[:, idx]
    # Box-plot per feature, scaled to [0,1] per column for visual comparability
    vmat_n = (vmat - vmat.min(axis=0)) / (vmat.max(axis=0) - vmat.min(axis=0) + 1e-8)
    ax.boxplot([vmat_n[:, k] for k in range(vmat_n.shape[1])], showfliers=False)
    ax.set_title(f'Block {block} ({len(idx)} features) — min-max-normalized box plot')
    ax.set_xticklabels([names[i].split('__')[1] for i in idx], rotation=90, fontsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
fig_path = OUT_DIR / 'pocket3d_distributions.png'
plt.savefig(fig_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'Saved → {fig_path}')

In [ ]:
# ── Cell 8: Final summary ───────────────────────────────────────────────────────────────────────────────────────
n_total = len(d['slugs'])
n_pocket = int(d['has_pocket'].sum())
ac_mean = float(d['atom_completeness'][d['has_pocket'] == 1].mean()) if n_pocket else 0.0

print(f'Pocket-3D extraction summary')
print(f'  total slugs            : {n_total}')
print(f'  has_pocket             : {n_pocket}  ({100*n_pocket/n_total:.1f}%)')
print(f'  atom_completeness mean : {ac_mean:.3f}')
print(f'  feature dim            : {X.shape[1]}')
print(f'  output                 : {out_path}')

# Acceptance gate per plan: has_pocket >= 850
if n_pocket >= 850:
    print('\n✓ Verification gate PASSED (has_pocket >= 850)')
    print('  Next: open validate_pocket3d.ipynb for sanity + mini-XGBoost gate')
else:
    print('\n✗ Verification gate FAILED (has_pocket < 850)')
    print('  Inspect failed slugs and consider falling back to grafted_fixed PDBs')